In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

from sklearn.base import BaseEstimator , TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import cross_validate
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split , RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from tqdm import tqdm
from dataclasses import dataclass

import scipy
#from /run/media/drdrakken/Elements/Sonstiges/Programmieren/Machine Learning/Part 13 - ProjectFolder/A1/Utillitys/MLUtilitys.py import MLUtilitys
#from Utillitys.MLUtilitys import MLUtilitys

In [ ]:
linux_path = r"/run/media/drdrakken/Elements/Sonstiges/Programmieren/Machine Learning/csvs/credit_fraud_detection/archive.zip"
windows_path = r""
df = pd.read_csv(linux_path)

In [ ]:
@dataclass
class Config:
    test_size: float = 0.2
    seed: int = 1234
    target: str = "Class"
    cross_iteration: int = 3
    grid_iterations: int = 3
    verbose : int = 0
    
config = Config()

In [ ]:
class DataClass():

    def __init__(self):
        
        self.x = df.drop([config.target] , axis = 1)
        self.y = df[config.target]

        self.numerical_data = self.x.select_dtypes(include = "number").columns
        self.categorical_data  = self.x.select_dtypes(exclude = "number").columns
        
data = DataClass()

In [ ]:
class DataTransform(BaseEstimator , TransformerMixin):
    def fit(self , X , y = None):
        return self

    def transform(self, X , y = None):
        x = X.copy()
        x = self.NewFeatures(x)
        x = self.time_features(x)
        x = self.zscsore(x)
        x = self.drop_cols(x)
        return x
    
    def NewFeatures(self , X):
        x = X.copy()

        return x
    
    def time_features(self , X):
        x = X.copy()
        x["Hours"] = (x["Time"] // 3600) % 24
        x["Days"] = x["Time"] // (3600 * 24)
        x["IsWeekendDay"] = (x["Days"] % 7 >= 5).astype(int)
        x["NightTime"] = x["Hours"].between(0 , 4).astype(int)
       # x["Last10Transactions"] = x["Amount"].rolling(window = 10).mean() # potential dataleakage

        timr_bins = [0 , 6 , 12 , 18 , 24]
        time_labels = ["Night" , "Morning" , "Evening" , "AfterNoon"]

        x["Time_period"] = pd.cut(x["Hours"] , bins = timr_bins , labels = time_labels , right = False)

        return x
    
    def zscsore(self , X):
        x = X.copy()
        x["ZScore"] = (x["Amount"] - x["Amount"].mean()) / x["Time"].std()
        return x
    
    def drop_cols(self , X):
        x = X.copy()
        x = x.drop(["Time" , "Hours"] , axis = 1)
        return x

In [ ]:
class Visualize():

    def __init__(self , OrigData):
        self.data = OrigData.copy()

    def iqr(self, TargetCol):
        q1 = self.data[TargetCol].quantile(0.25)
        q3 = self.data[TargetCol].quantile(0.75)
        iqr = q3 - q1
        return q1 , q3 , iqr
    
    def skew(self , skew_col):
        data_skew = self.data[skew_col].skew()
        print(f"Skewness of Col{data_skew}")

    def plot(self):

        for vis in self.data.numerical_data:
            fig , axes = plt.subplots(3 , 1 , figsize = (10 , 10 ) , dpi = 200)
            q1 , q3 , _ = self.iqr(TargetCol = vis)
            mean = self.data[vis].mean()
            self.skew(skew_col = vis)

            sns.histplot(data = self.data , x = vis , ax = axes[0])
            axes[0].axvline(q1 , color = "green")
            axes[0].axvline(q1 , color = "red")
            axes[0].axvline(mean , color = "yellow")
            axes[0].set_title(f"Histplot for:{vis}")

            sns.boxplot(data = self.data , X = vis , ax = axes[1])
            axes[1].axvline(q1 , color = "green")
            axes[1].axvline(q3 , color = "red")
            axes[1].axvline(mean , color = "yellow")
            axes[1].set_title(f"boxplot for:{vis}")

            sns.scatterplot(data = self.data , X = vis , y = self.data.y ,ax = axes[2])
            axes[2].axvline(q1 , color = "green")
            axes[2].axvline(q3 , color = "red")
            axes[2].axvline(mean , color = "yellow")
            axes[2].set_title(f"scatterplot for:{vis}")

            plt.tight_layout()
            plt.show()

In [ ]:
class Preprocessor():

    def fit(self, X , y = None):

        self.numerical_data = X.select_dtypes(include = np.number).columns
        self.categorical_data = X.select_dtypes(exclude = np.number).columns

        self.preprocessor = ColumnTransformer([
            ("numerical_pipe" , Pipeline([
                ("numerical_imputer" , SimpleImputer(strategy = "mean")),
                ("numerical_scaler" , MinMaxScaler())
            ]),self.numerical_data),

            ("categorical_pipeline" , Pipeline([
                ("categorical_imputer" , SimpleImputer(strategy="most_frequent")),
                ("categorical_encoder" , OneHotEncoder(handle_unknown="ignore" , sparse_output=False))
            ]),self.categorical_data)
        ])
        self.preprocessor.fit(X)
        return self

    def transform(self, X):
        return self.preprocessor.transform(X)

In [ ]:
def custom_pipeline(estimator , transform = False):

    steps = []
    if transform:
        steps.append(("transformed", DataTransform()))  

    steps.append(("base_performance" , Preprocessor()))
    steps.append(("estimator", estimator))
    return Pipeline(steps) 


In [ ]:
def model_varianz():
    return {
        "LogisticRegression" : LogisticRegression(max_iter=5000),
        "DecisionTreeClassifier" : DecisionTreeClassifier(random_state = config.seed),
        "RandomForestClassifier" : RandomForestClassifier(random_state = config.seed , n_jobs = -1),
        "LinearSVC": LinearSVC(random_state = config.seed , max_iter = 1000),
    }

In [ ]:
def split_data():
    X_train , X_test , y_train  , y_test = train_test_split(data.x,
                                                            data.y,
                                                            random_state = config.seed,
                                                            shuffle = True,
                                                            stratify = data.y,
                                                            test_size = config.test_size)
    return X_train , X_test , y_train  , y_test

In [ ]:
def custom_scoring_dict():
    return {
        "roc_auc":"roc_auc",
        "f1":"f1",
        "pr_auc":"average_precision",
        }

In [ ]:
def custom_data_validation(estimator , X_train , y_train , scoring_dict):
    kfold = StratifiedKFold(n_splits = 3 , shuffle = True , random_state = config.seed)
 
    return cross_validate(estimator = estimator,
                          X = X_train,
                          y = y_train,
                          return_train_score=True ,
                          return_estimator= True,
                          scoring = scoring_dict,
                          cv = kfold,
                          n_jobs= -1,
                          verbose=config.verbose,
                          )

In [ ]:
def custom_grid_search(estimator ,X_train , y_train , param_grid):
    grid = RandomizedSearchCV(estimator=estimator,
                              param_distributions=param_grid,
                              return_train_score=True,
                              refit="accuracy",
                              cv = config.cross_iteration,
                              verbose=config.verbose,
                              n_iter=config.cross_iteration,
                              random_state=config.seed,
                              )
    
    grid.fit(X_train , y_train)
    return grid

In [ ]:
def custom_parameter_grid():
        
        return {

            "LogisticRegression": {
                "estimator__C": scipy.stats.loguniform(1e-4, 1e2),
                "estimator__penalty": ["l2"],
                "estimator__solver": ["lbfgs"],
            },

            "DecisionTreeClassifier": {
                "estimator__criterion": ["gini", "entropy"],
                "estimator__max_depth": [None, 5, 10, 20],
                "estimator__min_samples_split": scipy.stats.randint(2, 20),
                "estimator__min_samples_leaf": scipy.stats.randint(1, 10),
            },

            "RandomForestClassifier": {
                "estimator__n_estimators": scipy.stats.randint(100, 500),
                "estimator__max_depth": scipy.stats.randint(5, 40),
                "estimator__min_samples_split": scipy.stats.randint(2, 20),
                "estimator__min_samples_leaf": scipy.stats.randint(1, 10),
                "estimator__max_features": ["sqrt", "log2"],
                "estimator__bootstrap": [True, False],
            },

            "LinearSVC": {
                "estimator__C": scipy.stats.loguniform(1e-4, 1e2),
                "estimator__loss": ["hinge", "squared_hinge"],
                "estimator__dual": [True],
                "estimator__max_iter": [5000, 10000],
            },

            "XGBClassifier": {
                "estimator__n_estimators": scipy.stats.randint(100, 500),
                "estimator__learning_rate": scipy.stats.loguniform(1e-3, 0.3),
                "estimator__max_depth": scipy.stats.randint(3, 10),
                "estimator__subsample": scipy.stats.uniform(0.6, 0.4),
                "estimator__colsample_bytree": scipy.stats.uniform(0.6, 0.4),
                "estimator__gamma": scipy.stats.uniform(0, 5),
                "estimator__min_child_weight": scipy.stats.randint(1, 10),
            },

        }

In [ ]:
class Benchmark():

    def __init__(self , UseCV = False, UseGridSearch = False):
        self.X_train , self.X_test , self.y_train  , self.y_test = split_data()
        self.results = []

        self.use_cv = UseCV
        self.use_grid_search = UseGridSearch
        self.train()
        

    def train(self):
        for estimator_name , estimator in model_varianz().items():

            base_pipe = custom_pipeline(estimator=estimator , transform=False)
            transformed_pipe = custom_pipeline(estimator=estimator , transform=True)

            if self.use_cv:
                base_cv = custom_data_validation(estimator=base_pipe,
                                                 X_train=self.X_train,
                                                 y_train=self.y_train,
                                                 scoring_dict=custom_scoring_dict())

                transformed_cv = custom_data_validation(estimator=transformed_pipe,
                                                 X_train=self.X_train,
                                                 y_train=self.y_train,
                                                 scoring_dict=custom_scoring_dict())
                
                self.results.append({

                    "Base_cv_train_roc_auc":base_cv["train_roc_auc"].mean(),
                    "Base_cv_test_roc_auc":base_cv["test_roc_auc"].mean(),
                    "Base_cv_std_test_roc_auc":base_cv["test_roc_auc"].std(),

                    "Base_cv_train_f1":base_cv["train_f1"].mean(),
                    "Base_cv_test_f1":base_cv["test_f1"].mean(),
                    "Base_cv_std_test_roc_auc":base_cv["test_roc_auc"].std(),

                    "Base_cv_train_pr_auc":base_cv["train_pr_auc"].mean(),
                    "Base_cv_test_pr_auc":base_cv["test_pr_auc"].mean(),
                    "Base_cv_std_test_roc_auc":base_cv["test_roc_auc"].std(),

                })

                self.results.append({

                    "Base_cv_train_roc_auc":transformed_cv["train_roc_auc"].mean(),
                    "Base_cv_test_roc_auc":transformed_cv["test_roc_auc"].mean(),
                    "Base_cv_std_test_roc_auc":transformed_cv["test_roc_auc"].std(),

                    "Base_cv_train_f1":transformed_cv["train_f1"].mean(),
                    "Base_cv_test_f1":transformed_cv["test_f1"].mean(),
                    "Base_cv_std_test_roc_auc":transformed_cv["test_roc_auc"].std(),

                    "Base_cv_train_pr_auc":transformed_cv["train_pr_auc"].mean(),
                    "Base_cv_test_pr_auc":transformed_cv["test_pr_auc"].mean(),
                    "Base_cv_std_test_roc_auc":transformed_cv["test_roc_auc"].std(),

                })

            if self.use_grid_search:
                base_grid = custom_grid_search(estimator=base_pipe,
                                            X_train=self.X_train,
                                            y_train=self.y_train,
                                            param_grid=custom_parameter_grid()[estimator_name])

                                          

                transformed_grid = custom_grid_search(estimator=base_pipe,
                                            X_train=self.X_train,
                                            y_train=self.y_train,
                                            param_grid=custom_parameter_grid()[estimator_name] 
                                            )

                self.results.append({
                    "Base_cv_train_score":base_grid.best_estimator_,
                    "Base_cv_train_score":base_grid.best_params_,
                })

                self.results.append({
                    "Base_cv_train_score":base_grid.best_estimator_,
                    "Base_cv_train_score":transformed_grid.best_params_,
                })

        self.performance_heatmap()
        
    def performance_heatmap(self, df):

        metrics = [
            "roc_score",
            "f1_score",
            "pr_score",
            "tuned_roc_score",
            "tuned_f1_score",
            "tuned_pr_score"
        ]

        heatmap_data = df.set_index("estimator")[metrics]

        plt.figure(figsize=(12, 8))

        plt.imshow(
            heatmap_data,
            aspect="auto"
        )

        plt.xticks(
            range(len(metrics)),
            metrics,
            rotation=45,
            ha="right"
        )

        plt.yticks(
            range(len(heatmap_data)),
            heatmap_data.index
        )

        plt.colorbar(label="Score")

        plt.title("Model Performance Heatmap")

        plt.tight_layout()
        plt.show()


In [ ]:
Benchmark(UseCV=True , UseGridSearch=True)

/home/drdrakken/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/drdrakken/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/drdrakken/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid thi